# Since at first, I didn't have matlab, I solved using python. but now both python code and matlab code has been provided

In [99]:
import numpy as np
import math
import sympy as sp
from scipy.optimize import root
from scipy.io import savemat

In [78]:
def get_a(a: float, b: float, theta: sp.Symbol):
    return [
        a * sp.cos(theta),
        a * sp.sin(theta),
        b
    ]
def get_Q(alpha: float, theta: sp.Symbol):
    return [
        [sp.cos(theta), -sp.cos(alpha) * sp.sin(theta), sp.sin(alpha) * sp.sin(theta)],
        [sp.sin(theta), sp.cos(alpha) * sp.cos(theta), -sp.sin(alpha) * sp.cos(theta)],
        [0, sp.sin(alpha), sp.cos(alpha)]
    ]

In [79]:
t1, t2, t3, t4 = sp.symbols("theta1 theta2 theta3 theta4")
print(get_a(5, 10, 1 - t1))
print(get_Q(sp.pi / 2, 1 - t1))

[5*cos(theta1 - 1), -5*sin(theta1 - 1), 10]
[[cos(theta1 - 1), 0, -sin(theta1 - 1)], [-sin(theta1 - 1), 0, -cos(theta1 - 1)], [0, 1, 0]]


In [ ]:
import sympy as sp
from dataclasses import dataclass
from typing import Union

@dataclass
class DHParameters:
    """DH Parameters for a robotic manipulator"""
    a: Union[float, sp.Symbol]
    b: Union[float, sp.Symbol]
    alpha: Union[float, sp.Symbol]
    theta: Union[float, sp.Symbol]
    
    def __repr__(self):
        return f"DH(a={self.a}, b={self.b}, alpha={self.α}, theta={self.θ})"
    
    @property
    def α(self):
        """Alias for alpha (using Greek letter)"""
        return self.alpha
    
    @property
    def θ(self):
        """Alias for theta (using Greek letter)"""
        return self.theta

class DHTable:
    """Denavit-Hartenberg Table Implementation"""
    
    def __init__(self):
        # Define symbolic joint variables
        self.theta1, self.theta2, self.theta3, self.theta4 = sp.symbols('θ1:5')
        # Initialize DH table according to your data
        self.table = [
            DHParameters(
                a=786.71,
                b=260,
                alpha=sp.pi/2,
                theta=self.theta1
            ),
            DHParameters(
                a=945,
                b=0,
                alpha=sp.pi,
                theta=self.theta2 - sp.pi/2
            ),
            DHParameters(
                a=1270.15,
                b=251.5,
                alpha=3*sp.pi/2,
                theta=self.theta3 - sp.pi/2
            ),
            DHParameters(
                a=0,
                b=0,
                alpha=0,
                theta=self.theta4 + sp.pi/2
            )
        ]
        
        # Store symbols for easy access
        self.symbols = {
            'θ1': self.theta1, 'θ2': self.theta2, 
            'θ3': self.theta3, 'θ4': self.theta4
        }
    
    def __getitem__(self, index: int) -> DHParameters:
        """Access DH parameters by index (0-based)"""
        return self.table[index]
    
    def __len__(self) -> int:
        """Number of joints"""
        return len(self.table)
    
    def __get_transformation_matrix(self, index: int) -> sp.Matrix:
        """Get the homogeneous transformation matrix for a given link"""
        params = self[index]
        
        # Extract parameters
        a, b, α, θ = params.a, params.b, params.alpha, params.theta
        
        # Build the DH transformation matrix
        # Rz(θ) * Tz(d) * Tx(a) * Rx(α)
        return sp.Matrix([
            [sp.cos(θ), -sp.sin(θ)*sp.cos(α), sp.sin(θ)*sp.sin(α), a*sp.cos(θ)],
            [sp.sin(θ), sp.cos(θ)*sp.cos(α), -sp.cos(θ)*sp.sin(α), a*sp.sin(θ)],
            [0, sp.sin(α), sp.cos(α), b],
            [0, 0, 0, 1]
        ])
    
    def get_rotation_matrix(self, index: int) -> sp.Matrix:
        T = self.__get_transformation_matrix(index)
        return T[:3, :3]
    
    def get_transfer_vector(self, index: int) -> sp.Matrix:
        T = self.__get_transformation_matrix(index)
        return T[:3, 3]

    
    def get_forward_kinematics(self, up_to: int = None, simplify: bool = True) -> sp.Matrix:
        """Calculate forward kinematics up to a specific joint"""
        if up_to is None:
            up_to = len(self) - 1
        
        T = sp.eye(4)
        for i in range(up_to + 1):
            T = T * self.__get_transformation_matrix(i)
        if simplify:
            T = sp.simplify(T)
        return T
    
    def get_jacobian(self) -> sp.Matrix:
        """Calculate the geometric Jacobian matrix"""
        raise NotImplementedError
    
    def calc_inverse(self, x: float, y: float, z: float, phi: float):
        forward = self.get_forward_kinematics()
        Q = forward[:3, :3]
        a = forward[:3, 3]

        phi_p = (sp.trace(Q) - 1 ) / 2
        equations = sp.Matrix([
            a[0] - x,
            a[1] - y,
            a[2] - z,
            phi_p - sp.cos(phi)
        ])
        symbols = self.symbols.values()
        equations = sp.lambdify(symbols, equations, 'numpy')
    
        def fun(theta_vals):
            return np.array(equations(*theta_vals), dtype=float).flatten()

        initial_guess = np.array([0.1, 0.1, 0.1, 0.1])

        sol = root(fun, initial_guess, method='lm')

        if sol.success:
            return sol.x
        else:
            raise RuntimeError(f"Inverse kinematics did not converge: {sol.message}")

    
    def display_table(self):
        """Pretty print the DH table"""
        print("DH Parameters Table:")
        print("Index |     a     |     b     |     α     |     θ     ")
        print("-" * 60)
        for i, params in enumerate(self.table):
            print(f"  {i}   | {params.a:9} | {params.b:9} | {sp.latex(params.alpha):9} | {sp.latex(params.theta)}")
    

dh = DHTable()
print(dh.get_rotation_matrix(1))
print(dh.get_transfer_vector(1))

Matrix([[sin(θ2), -cos(θ2), 0], [-cos(θ2), -sin(θ2), 0], [0, 0, -1]])
Matrix([[945*sin(θ2)], [-945*cos(θ2)], [0]])


In [81]:
for i in range(4):
    print(f"Q{i}")
    print(dh.get_rotation_matrix(i))

Q0
Matrix([[cos(θ1), 0, sin(θ1)], [sin(θ1), 0, -cos(θ1)], [0, 1, 0]])
Q1
Matrix([[sin(θ2), -cos(θ2), 0], [-cos(θ2), -sin(θ2), 0], [0, 0, -1]])
Q2
Matrix([[sin(θ3), 0, cos(θ3)], [-cos(θ3), 0, sin(θ3)], [0, -1, 0]])
Q3
Matrix([[-sin(θ4), -cos(θ4), 0], [cos(θ4), -sin(θ4), 0], [0, 0, 1]])


In [82]:
for i in range(4):
    print(f"a{i}")
    print(dh.get_transfer_vector(i))

a0
Matrix([[786.71*cos(θ1)], [786.71*sin(θ1)], [260]])
a1
Matrix([[945*sin(θ2)], [-945*cos(θ2)], [0]])
a2
Matrix([[1270.15*sin(θ3)], [-1270.15*cos(θ3)], [251.500000000000]])
a3
Matrix([[0], [0], [0]])


In [83]:
t = dh.get_forward_kinematics(simplify=False)
sp.simplify(t)

Matrix([
[ sin(θ1)*cos(θ4) - sin(θ4)*cos(θ1)*cos(θ2 - θ3), -sin(θ1)*sin(θ4) - cos(θ1)*cos(θ4)*cos(θ2 - θ3), sin(θ2 - θ3)*cos(θ1), -251.5*sin(θ1) + 945.0*sin(θ2)*cos(θ1) + 1270.15*cos(θ1)*cos(θ2 - θ3) + 786.71*cos(θ1)],
[-sin(θ1)*sin(θ4)*cos(θ2 - θ3) - cos(θ1)*cos(θ4), -sin(θ1)*cos(θ4)*cos(θ2 - θ3) + sin(θ4)*cos(θ1), sin(θ1)*sin(θ2 - θ3),  945.0*sin(θ1)*sin(θ2) + 1270.15*sin(θ1)*cos(θ2 - θ3) + 786.71*sin(θ1) + 251.5*cos(θ1)],
[                          -sin(θ4)*sin(θ2 - θ3),                           -sin(θ2 - θ3)*cos(θ4),        -cos(θ2 - θ3),                                           1270.15*sin(θ2 - θ3) - 945.0*cos(θ2) + 260.0],
[                                              0,                                               0,                    0,                                                                                      1]])

In [84]:

matlab_dict = {
    'T': str(t),  
    'theta1': 'theta1',
    'theta2': 'theta2',
    'theta3': 'theta3',
    'theta4': 'theta4'
}
savemat('matrix_symbolic.mat', matlab_dict)
print("Exported symbolic matrix as string")

Exported symbolic matrix as string


In [96]:

def solve_and_check_reverse(x, y, z, phi):
    res = dh.calc_inverse(x, y, z, phi)

    sub_dict = {
        dh.theta1: res[0],
        dh.theta2: res[1],
        dh.theta3: res[2],
        dh.theta4: res[3],
    }
    a = dh.get_forward_kinematics()[:3, 3]
    Q = dh.get_forward_kinematics()[:3, :3]
    phi_p = sp.acos((sp.trace(Q) - 1) / 2)
    print(a.subs(sub_dict))
    print(phi_p.subs(sub_dict ) / sp.pi)

In [98]:
print(sp.cos(2 * sp.pi / 3))
solve_and_check_reverse(1500, 15, 1450, 2 * sp.pi / 3)

-1/2
Matrix([[1499.99993678182], [14.9999993600105], [1450.00009608023]])
2.82352688573082/pi
